In [2]:
import io
import zipfile
import requests
import frontmatter

In [3]:
def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data


In [4]:
zoomcamp = read_repo_data('DataTalksClub', 'data-engineering-zoomcamp')
print(f"Found {len(zoomcamp)} documents")

Found 143 documents


In [5]:
import re

def split_markdown_by_level(text, level=2):
    """
    Split markdown text by a specific header level.
    
    :param text: Markdown text as a string
    :param level: Header level to split on
    :return: List of sections as strings
    """
    # This regex matches markdown headers
    # For level 2, it matches lines starting with "## "
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)

    # Split and keep the headers
    parts = pattern.split(text)
    
    sections = []
    for i in range(1, len(parts), 3):
        # We step by 3 because regex.split() with
        # capturing groups returns:
        # [before_match, group1, group2, after_match, ...]
        # here group1 is "## ", group2 is the header text
        header = parts[i] + parts[i+1]  # "## " + "Title"
        header = header.strip()

        # Get the content after this header
        content = ""
        if i+2 < len(parts):
            content = parts[i+2].strip()

        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header
        sections.append(section)
    
    return sections


In [6]:
zoomcamp_chunks = []

for doc in zoomcamp:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    sections = split_markdown_by_level(doc_content, level=2)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        zoomcamp_chunks.append(section_doc)

In [7]:
print(len(zoomcamp_chunks))
print(zoomcamp_chunks[0].keys())

620
dict_keys(['filename', 'section'])


In [8]:
from minsearch import Index

index = Index(
    text_fields=['section'],  # what fields do you have?
    keyword_fields=[]
)
index.fit(zoomcamp_chunks)

In [9]:
query = 'how do I setup Docker?'
results = index.search(query)
print(f"Number of results: {len(results)}")
print(results[0])

Number of results: 10
{'filename': 'data-engineering-zoomcamp-main/04-analytics-engineering/README.md', 'section': '## Setting up your environment\n\nChoose your setup path:\n\n### 🏠 [Local Setup](setup/local_setup.md)\n\n- **Stack**: DuckDB + dbt Core\n- **Cost**: Free\n- [→ Get Started](setup/local_setup.md)\n\n### ☁️ [Cloud Setup](setup/cloud_setup.md)\n\n- **Stack**: BigQuery + dbt Cloud\n- **Cost**: Free tier available (dbt Cloud Developer), BigQuery costs vary\n- **Requires**: Completed Module 3 with BigQuery data\n- [→ Get Started](setup/cloud_setup.md)'}


In [10]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
from minsearch import VectorSearch
from tqdm.auto import tqdm
import numpy as np

zcamp_embeddings = []

for record in tqdm(zoomcamp_chunks):
    text = record['section']
    v_doc = embedding_model.encode(text)
    zcamp_embeddings.append(v_doc)

zcamp_embeddings = np.array(zcamp_embeddings)

zcamp_vindex = VectorSearch()
zcamp_vindex.fit(zcamp_embeddings, zoomcamp_chunks)

  0%|          | 0/620 [00:00<?, ?it/s]

In [12]:
query = 'how do I setup Docker?'

# text search
text_results = index.search(query)
print("TEXT SEARCH:")
print(text_results[0]['section'][:200])

# vector search
q = embedding_model.encode(query)
vector_results = zcamp_vindex.search(q)
print("\nVECTOR SEARCH:")
print(vector_results[0]['section'])

TEXT SEARCH:
## Setting up your environment

Choose your setup path:

### 🏠 [Local Setup](setup/local_setup.md)

- **Stack**: DuckDB + dbt Core
- **Cost**: Free
- [→ Get Started](setup/local_setup.md)

### ☁️ [Clo

VECTOR SEARCH:
## Basic Docker Commands

Check Docker version:

```bash
docker --version
```

Run a simple container:

```bash
docker run hello-world
```

Run something more complex:

```bash
docker run ubuntu
```

Nothing happens. Need to run it in `-it` mode:

```bash
docker run -it ubuntu
```

We don't have `python` there so let's install it:

```bash
apt update && apt install python3
python3 -V
```


In [13]:
from typing import List, Any

def text_search(query: str, num_results: int = 3):
    return index.search(query, num_results=num_results)

def vector_search(query: str, num_results: int = 3):
    q = embedding_model.encode(query)
    return zcamp_vindex.search(q, num_results=num_results)

def hybrid_search(query: str) -> List[Any]:
    """
    Search the Zoomcamp course sections for relevant information.
    Args:
        query (str): The search query string.
    Returns:
        List[Any]: A list of search results from the course sections.
    """
    text_results = text_search(query, num_results=3)
    vector_results = vector_search(query, num_results=3)
    
    seen_ids = set()
    combined_results = []
    for result in text_results + vector_results:
        key = result['filename'] + result['section']
        if key not in seen_ids:
            seen_ids.add(key)
            # Truncate section to avoid token limits
            r = result.copy()
            r['section'] = result['section'][:500]
            combined_results.append(r)
    
    return combined_results


In [14]:
query = 'how do I setup Docker?'
print(f"Text: {len(text_search(query))} results")
print(f"Vector: {len(vector_search(query))} results")
print(f"Hybrid: {len(hybrid_search(query))} results")

Text: 3 results
Vector: 3 results
Hybrid: 6 results


In [15]:
from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv('../.env')
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [16]:
from pydantic_ai import Agent
from pydantic_ai.models.groq import GroqModel

model = GroqModel('llama-3.1-8b-instant')

system_prompt = """
You are a helpful assistant for a course. 
Use the search tool to find relevant information before answering questions.
If the first search doesn't give enough information, try different search terms.
"""

agent = Agent(
    model=model,
    system_prompt=system_prompt,
    tools=[hybrid_search]
)

In [17]:
question = "How long does it take to complete the course?"
result = await agent.run(user_prompt=question)
print(result.output)

<function=hybrid_search>{"query": "Course duration"}


In [18]:
question = "What tools do I need to install for module 1?"
result = await agent.run(user_prompt=question)
print(result.output)

Based on the search results, you will need the following tools for module 1:

- Docker
- PySpark

You can install Docker and PySpark as directed in the module 1 documentation and module 5 homework. 

However, let me clarify and provide you a more specific answer. Since the search results point to different modules and instructions, it seems that the module 1 might refer to Docker and Terraform as they are mentioned as required setup in module 1. Therefore, you should ensure that you have Docker and Terraform set up properly.


In [19]:
from pydantic_ai.messages import ModelMessagesTypeAdapter

def log_entry(agent, messages, source="user"):
    tools = []
    for ts in agent.toolsets:
        tools.extend(ts.tools.keys())

    dict_messages = ModelMessagesTypeAdapter.dump_python(messages)

    system_prompt = agent._instructions
    if not system_prompt:
        try:
            system_prompt = dict_messages[0]['parts'][0]['content']
        except:
            system_prompt = ""

    return {
        "agent_name": agent.name,
        "system_prompt": system_prompt,
        "provider": agent.model.system,
        "model": agent.model.model_name,
        "tools": tools,
        "messages": dict_messages,
        "source": source
    }

In [20]:
import json
import secrets
from pathlib import Path
from datetime import datetime


LOG_DIR = Path('logs')
LOG_DIR.mkdir(exist_ok=True)


def serializer(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    raise TypeError(f"Type {type(obj)} not serializable")


def log_interaction_to_file(agent, messages, source='user'):
    entry = log_entry(agent, messages, source)

    ts = entry['messages'][-1]['timestamp']
    # ts_obj = datetime.fromisoformat(ts.replace("Z", "+00:00"))
    ts_str = ts.strftime("%Y%m%d_%H%M%S")
    rand_hex = secrets.token_hex(3)

    filename = f"{agent.name}_{ts_str}_{rand_hex}.json"
    filepath = LOG_DIR / filename

    with filepath.open("w", encoding="utf-8") as f_out:
        json.dump(entry, f_out, indent=2, default=serializer)

    return filepath


In [21]:
question = "How do I set up Docker for the course?"
result = await agent.run(user_prompt=question)
print(result.output)
log_interaction_to_file(agent, result.new_messages())

Based on the search results, it appears that the course is providing guidance on setting up Docker, using Docker Compose, and understanding the benefits of Docker. 

To set up Docker for the course, you can start by following the instructions provided in the search results. However, without more specific information, it's difficult to provide a step-by-step guide. 

Here is a possible answer:

To set up Docker for the course, start by installing Docker on your local machine, if you haven't already done so. You can then follow the instructions in the file "data-engineering-zoomcamp-main/01-docker-terraform/docker-sql/09-docker-compose.md" to set up Docker Compose. This will allow you to start and manage containers for the services required for the course. 

Additionally, you can practice using Docker by following the homework instructions provided in the file "data-engineering-zoomcamp-main/cohorts/2023/week_1_docker_sql/homework.md". This will give you hands-on experience with Docker a

PosixPath('logs/agent_20260331_010704_d0b104.json')

In [22]:
import time

questions = [
    "What tools do I need to install for module 1?",
    "How do I set up BigQuery?",
    "What is the homework for the Kafka module?",
    "How do I connect Spark to GCS?",
    "What are the prerequisites for the course?"
]

for q in tqdm(questions):
    print(q)
    result = await agent.run(user_prompt=q)
    print(result.output)
    log_interaction_to_file(agent, result.new_messages())
    print()
    time.sleep(30)

  0%|          | 0/5 [00:00<?, ?it/s]

What tools do I need to install for module 1?
For module 1, you will need to install the following tools:

- Docker
- Python
- Git (optional)

How do I set up BigQuery?
<function=hybrid_search>{"query": "bigquery setup"}

What is the homework for the Kafka module?
The homework for the Kafka module involves extending Module 5 Homework and learning about streaming with PySpark using Red Panda, which is a drop-in replacement for Kafka.

How do I connect Spark to GCS?
To connect Spark to GCS, you need to follow the instructions below:

1. Install Spark and PySpark on your machine.
2. Run PySpark and create a local Spark session.
3. Download the jar for connecting to GCS from the Google Cloud Console.
4. Copy the downloaded jar to a location on your machine (e.g., the `lib` folder).
5. Upload the required data to GCS using the `gsutil` command.
6. Connect to GCS by specifying the GCS connector in the Spark configuration.

The exact steps may vary depending on your operating system and the v

In [23]:
import random 

sample = random.sample(zoomcamp_chunks, 10)
prompt_docs = [d['section'] for d in sample] 

In [24]:
evaluation_prompt = """
Use this checklist to evaluate the quality of an AI agent's answer (<ANSWER>) to a user question (<QUESTION>).
We also include the entire log (<LOG>) for analysis.

For each item, check if the condition is met. 

Checklist:

- instructions_follow: The agent followed the user's instructions (in <INSTRUCTIONS>)
- instructions_avoid: The agent avoided doing things it was told not to do  
- answer_relevant: The response directly addresses the user's question  
- answer_clear: The answer is clear and correct  
- answer_citations: The response includes proper citations or sources when required  
- completeness: The response is complete and covers all key aspects of the request
- tool_call_search: Is the search tool invoked? 

Output true/false for each check and provide a short explanation for your judgment.
""".strip()

In [25]:
from pydantic import BaseModel

class EvaluationCheck(BaseModel):
    check_name: str
    justification: str
    check_pass: bool

class EvaluationChecklist(BaseModel):
    checklist: list[EvaluationCheck]
    summary: str


In [45]:
eval_agent = Agent(
    name='eval_agent',
    model=GroqModel('llama-3.3-70b-versatile'),
    instructions=evaluation_prompt + "\n\nReturn ONLY a JSON object with keys 'checklist' and 'summary'. No tags, no markdown."
)

In [27]:
system_prompt = """
You are a helpful assistant for the Data Engineering Zoomcamp course.
Use the search tool to find relevant information before answering questions.
If the first search doesn't give enough information, try different search terms.

Always include references by citing the filename of the source material you used.
When citing the reference, replace "data-engineering-zoomcamp-main" with the full path to the GitHub repository: "https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/"
Format: [LINK TITLE](FULL_GITHUB_LINK)

If the search doesn't return relevant results, let the user know and provide general guidance.
""".strip()

agent = Agent(
    name="zoomcamp_agent",
    model=model,
    system_prompt=system_prompt,
    tools=[hybrid_search]
)

In [28]:
result = await agent.run(user_prompt="How do I set up Docker?")
print(result.output)

To set up Docker, you can start by checking the Docker version using the command `docker --version` [1](https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/01-docker-terraform/docker-sql/01-introduction.md). Then, you can run a simple container using the command `docker run hello-world` [1](https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/01-docker-terraform/docker-sql/01-introduction.md). If you want to run a more complex container, you can use the command `docker run -it ubuntu` [1](https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/01-docker-terraform/docker-sql/01-introduction.md).

To run a container interactively, you can use the `-it` flag with the `docker run` command. For example, to run a Python 3.9 image, you can use the command `docker run -it python:3.9 bash` [2](https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/cohorts/2024/01-docker-terraform/solutions.md).

You can also use Docker Compose to start

In [29]:
question_generation_prompt = """
You are helping to create test questions for an AI agent that answers questions about a data engineering zoomcamp.

Based on the provided content, generate realistic questions that students might ask.

The questions should:

- Be natural and varied in style
- Range from simple to complex
- Include both specific technical questions and general course questions

Generate one question for each record.
""".strip()

class QuestionsList(BaseModel):
    questions: list[str]

question_generator = Agent(
    name="question_generator",
    instructions=question_generation_prompt,
    model= GroqModel('llama-3.3-70b-versatile'),
    output_type=QuestionsList
)


In [30]:
import random
import time

sample = random.sample(zoomcamp_chunks, 10)
prompt_docs = [d['section'] for d in sample]
prompt = json.dumps(prompt_docs)

result = await question_generator.run(prompt)
questions = result.output.questions

for q in tqdm(questions):
    print(q)
    result = await agent.run(user_prompt=q)
    print(result.output)
    log_interaction_to_file(agent, result.new_messages(), source='ai-generated')
    print()
    time.sleep(30)

  0%|          | 0/9 [00:00<?, ?it/s]

What should I do if the 'File /.google/credentials/google_credentials.json was not found' error occurs while setting up Airflow?
To resolve the 'File /.google/credentials/google_credentials.json was not found' error while setting up Airflow, follow these steps:

1. Make sure you have your credentials in your `$HOME/.google/credentials`. Check if you missed the step and didn't copy your JSON with credentials there. Also, ensure that the file name is `google_credentials.json`.
2. Check that docker-compose can correctly map this directory to airflow worker.
3. Execute `docker ps` to see the list of docker containers running on your host machine and find the ID of the relevant container.

Additionally, if you're running Docker in Windows/WSL/WSL2, you might need to check the airflow & WSL2 gist for troubleshooting possible problems. [LINK TITLE](https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/cohorts/2022/week_2_data_ingestion/airflow/2_setup_nofrills.md)

How can I re

In [32]:
def load_log_file(log_file):
    with open(log_file, 'r') as f_in:
        log_data = json.load(f_in)
        log_data['log_file'] = log_file
        return log_data

In [41]:
def simplify_log_messages(messages):
    log_simplified = []
    for m in messages:
        parts = []
        for original_part in m['parts']:
            part = original_part.copy()
            kind = part['part_kind']
            if kind == 'user-prompt':
                part.pop('timestamp', None)
            if kind == 'tool-call':
                part.pop('tool_call_id', None)
            if kind == 'tool-return':
                part.pop('tool_call_id', None)
                part.pop('metadata', None)
                part.pop('timestamp', None)
                part['content'] = 'RETURN_RESULTS_REDACTED'
            if kind == 'text':
                part.pop('id', None)
            parts.append(part)
        log_simplified.append({'kind': m['kind'], 'parts': parts})
    return log_simplified

In [38]:
user_prompt_format = """
<INSTRUCTIONS>{instructions}</INSTRUCTIONS>
<QUESTION>{question}</QUESTION>
<ANSWER>{answer}</ANSWER>
<LOG>{log}</LOG>
""".strip()

In [55]:
async def evaluate_log_record(eval_agent, log_record):
    messages = log_record['messages']
    instructions = log_record['system_prompt']
    question = messages[0]['parts'][0]['content']
    answer = messages[-1]['parts'][0]['content']
    log_simplified = simplify_log_messages(messages)
    log = json.dumps(log_simplified)

    user_prompt = user_prompt_format.format(
        instructions=instructions,
        question=question,
        answer=answer,
        log=log
    )

    result = await eval_agent.run(user_prompt)
    raw = result.output
    raw = re.sub(r'<function[^>]*>', '', raw)
    raw = re.sub(r'</function>', '', raw).strip()
    
    json_match = re.search(r'\{.*\}', raw, re.DOTALL)
    if json_match:
        data = json.loads(json_match.group())
        
        # Handle checklist as dict: {"checklist": {"instructions_follow": true, ...}}
        if 'checklist' in data and isinstance(data['checklist'], dict):
            checklist = [
                EvaluationCheck(
                    check_name=k,
                    justification="",
                    check_pass=v
                )
                for k, v in data['checklist'].items()
                if isinstance(v, bool)
            ]
            
            # handle summary being either a string or a dict
            summary = data.get('summary', '')
            if isinstance(summary, dict):
                summary = summary.get('explanation', '')
            
            return EvaluationChecklist(
                checklist=checklist,
                summary=summary
            )
        
        # Handle flat format: {"instructions_follow": true, ...}
        if 'checklist' not in data:
            checklist = [
                EvaluationCheck(
                    check_name=k,
                    justification="",
                    check_pass=v
                )
                for k, v in data.items()
                if isinstance(v, bool)
            ]
            return EvaluationChecklist(checklist=checklist, summary="")
    
    return EvaluationChecklist(**data)

In [54]:
log_record = load_log_file(list(LOG_DIR.iterdir())[0])
result = await eval_agent.run(user_prompt_format.format(
    instructions=log_record['system_prompt'],
    question=log_record['messages'][0]['parts'][0]['content'],
    answer=log_record['messages'][-1]['parts'][0]['content'],
    log=json.dumps(simplify_log_messages(log_record['messages']))
))
import json, re
raw = result.output
raw = re.sub(r'<function[^>]*>', '', raw)
raw = re.sub(r'</function>', '', raw).strip()
json_match = re.search(r'\{.*\}', raw, re.DOTALL)
data = json.loads(json_match.group())
print(json.dumps(data, indent=2))

{
  "checklist": {
    "instructions_follow": true,
    "instructions_avoid": true,
    "answer_relevant": true,
    "answer_clear": false,
    "answer_citations": false,
    "completeness": false,
    "tool_call_search": true
  },
  "summary": {
    "explanation": "The agent followed instructions by using the search tool, avoided doing things it was told not to do, and the search query is relevant to the user's question. However, the answer is not clear and does not provide a direct solution to the user's question, lacks proper citations, and does not cover all key aspects of the request."
  }
}


In [56]:
eval_set = []

for log_file in LOG_DIR.iterdir():
    log_record = load_log_file(log_file)
    if log_record['source'] != 'ai-generated':
        continue
    eval_set.append(log_record)

print(f"Found {len(eval_set)} ai-generated logs")

eval_results = []

for log_record in tqdm(eval_set):
    eval_result = await evaluate_log_record(eval_agent, log_record)
    eval_results.append((log_record, eval_result))
    time.sleep(10)  # avoid rate limits on eval agent too

Found 9 ai-generated logs


  0%|          | 0/9 [00:00<?, ?it/s]

In [58]:
import pandas as pd

rows = []
for log_record, eval_result in eval_results:
    messages = log_record['messages']
    row = {
        'question': messages[0]['parts'][0]['content'],
        'answer': messages[-1]['parts'][0]['content'],
    }
    checks = {c.check_name: c.check_pass for c in eval_result.checklist}
    row.update(checks)
    rows.append(row)

df_evals = pd.DataFrame(rows)
print(df_evals.mean(numeric_only=True))

instructions_follow    0.888889
instructions_avoid     1.000000
answer_relevant        0.777778
answer_clear           1.000000
answer_citations       1.000000
completeness           0.666667
tool_call_search       1.000000
dtype: float64
